In [1]:
import sys
import os

target_dir = os.path.abspath('..') 

if target_dir not in sys.path:
    sys.path.append(target_dir)

In [ ]:
# Use helpers
from preprocess import * 
from understatapi import UnderstatClient

from preprocess.player_stats import get_position_players_stats_df

understat = UnderstatClient()

# Fit model to forwards for proof of concept
stats = ["goals", "xG", "assists", "xA", "key_passes", "xGChain", "xGBuildup"]
window_size = 10

f_stats = get_position_players_stats_df(understat, ['F'], stats)

In [3]:
# train/test split
train_len = int(len(f_stats) * 0.8)
train_df = f_stats.iloc[:train_len]
test_df = f_stats.iloc[train_len:]

In [4]:
from preprocess.player_stats import *

# create datasets
train_dataset = CustomFootballDataset(train_df, window_size, multiple_players=False)
test_dataset = CustomFootballDataset(test_df, window_size, multiple_players=False)

In [5]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False
)

In [6]:
from football_lstm import *
import torch.optim as optim

model = FootballLSTM(n_features=len(f_stats.columns), hidden_size=32)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 20

model.train_model(
    optimizer=optimizer,
    loss_fn=loss_fn,
    train_dataloader=train_dataloader,
    n_epochs=num_epochs
)